<a href="https://colab.research.google.com/github/Areeha548/Text_Preprocessing_NLP/blob/main/Areeha_Shafique_Text_Preprocessing_NLP_Solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Preprocessing for NLP — Lecture & Live Demo

A runnable notebook. Each section first **explains** a step, then **demonstrates** it in code you can run immediately.
English and Urdu examples run in parallel throughout.

**How to use:** run the cells top to bottom. The demos use only `numpy` and `scikit-learn` plus pure-Python helpers,
so they run instantly with no downloads. `nltk` is shown as an optional upgrade where relevant.

---

## 0. Setup

This cell imports what we need and detects optional libraries. Nothing here needs the internet.

In [1]:
import re, unicodedata
from collections import Counter

try:
    import numpy as np
    HAS_NUMPY = True
except Exception:
    HAS_NUMPY = False

try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from sklearn.preprocessing import OneHotEncoder
    HAS_SK = True
except Exception:
    HAS_SK = False

# nltk is optional. If present we use it; if not, pure-Python fallbacks run instead.
try:
    import nltk
    for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

print('numpy:', HAS_NUMPY, '| sklearn:', HAS_SK, '| nltk:', HAS_NLTK)

numpy: True | sklearn: True | nltk: True


## 1. Why preprocess?

Models read **numbers**, not text. Preprocessing turns messy human language into clean, consistent tokens,
then into vectors. The principle is *garbage in, garbage out*: `Good`, `good`, and `GOOD` should count as one word.

The pipeline we build: **clean → lowercase/normalize → tokenize → remove stop words → stem/lemmatize → vectors**.

**Urdu note.** Urdu is right-to-left, has no letter case, uses optional diacritics, and inflects heavily,
so it breaks English assumptions and makes a good stress test for any pipeline.

## 2. Cleaning & normalization

Real text carries HTML, URLs, emails, and inconsistent spacing. Clean it first.
For Urdu, the parallel step is **character normalization**: unify letter variants and strip optional diacritics (*aeraab*).

In [2]:
def clean(text):
    text = re.sub(r'<[^>]+>', ' ', text)          # HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)  # URLs
    text = re.sub(r'\S+@\S+', ' ', text)           # emails
    text = re.sub(r'\s+', ' ', text).strip()        # collapse whitespace
    return text

print(clean('Visit https://x.com! Email a@b.com  <b>NOW</b>'))

Visit Email NOW


In [3]:
# --- Urdu normalization (pure Python, always runs) ---
ARABIC_TO_URDU = {'\u064a': '\u06cc', '\u0643': '\u06a9', '\u0649': '\u06cc'}  # yeh, kaf, alef-maksura
DIACRITICS = '\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652'  # aeraab marks

def urdu_normalize(text):
    text = ''.join(ARABIC_TO_URDU.get(ch, ch) for ch in text)   # unify letter variants
    text = ''.join(ch for ch in text if ch not in DIACRITICS)   # strip diacritics
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw = '\u0645\u064e\u062f\u0631\u064e\u0633\u064e\u06c1 \u0645\u06cc\u06ba \u0644\u0691\u06a9\u06d2'  # madrasah mein larke (with diacritics)
print('raw       :', raw)
print('normalized:', urdu_normalize(raw))

raw       : مَدرَسَہ میں لڑکے
normalized: مدرسہ میں لڑکے


## 3. Lowercasing

Fold case so `Apple`, `apple`, `APPLE` become one token. English-only — Urdu has no case, so it is skipped
(normalization from Section 2 plays that role).

In [4]:
s = 'The Cat SAT on the Mat'
print(s.lower())

the cat sat on the mat


## 4. Tokenization

Split text into tokens. Whitespace splitting is a weak start — it glues punctuation on and mishandles contractions.
For Urdu, spacing is unreliable, so a real pipeline uses a trained segmenter; here we use normalization + split as a demo.

In [5]:
text = "I don't love NLP."
print('naive :', text.split())
print('regex :', re.findall(r'\w+|[^\w\s]', text))

if HAS_NLTK:
    try:
        from nltk.tokenize import word_tokenize
        print('nltk  :', word_tokenize(text))
    except Exception as e:
        print('nltk tokenizer data unavailable:', e)

naive : ['I', "don't", 'love', 'NLP.']
regex : ['I', 'don', "'", 't', 'love', 'NLP', '.']
nltk  : ['I', 'do', "n't", 'love', 'NLP', '.']


In [6]:
def urdu_tokenize(text):
    return urdu_normalize(text).split()

ur = '\u0622\u062c \u0645\u0648\u0633\u0645 \u0628\u06c1\u062a \u0627\u0686\u06be\u0627 \u06c1\u06d2'  # 'the weather is very nice today'
print(urdu_tokenize(ur))

['آج', 'موسم', 'بہت', 'اچھا', 'ہے']


## 5. Stop-word removal

Drop frequent, low-information words so content words dominate. Lists are language-specific:
an English list removes nothing from Urdu. Skip this step when a dropped word carries signal (e.g. *not* in sentiment).

In [7]:
EN_STOP = {'i','the','of','is','a','an','and','to','in','on','are','was'}
UR_STOP = {'\u06a9\u06d2','\u06a9\u0627','\u06a9\u06cc','\u06a9\u0648','\u0633\u06d2','\u0645\u06cc\u06ba','\u067e\u0631',
           '\u06c1\u06d2','\u06c1\u06cc\u06ba','\u062a\u06be\u0627','\u062a\u06be\u06cc','\u062a\u06be\u06d2','\u0627\u0648\u0631',
           '\u06cc\u06c1','\u0648\u06c1','\u0631\u06c1\u06d2','\u0646\u06d2'}

eng = ['i','love','the','game','of','cricket']
urd = urdu_tokenize('\u0645\u06cc\u06ba \u0646\u06d2 \u06a9\u0631\u06a9\u0679 \u06a9\u0627 \u06a9\u06be\u06cc\u0644 \u062f\u06cc\u06a9\u06be\u0627')
print('EN:', [w for w in eng if w not in EN_STOP])
print('UR:', [w for w in urd if w not in UR_STOP])

EN: ['love', 'game', 'cricket']
UR: ['کرکٹ', 'کھیل', 'دیکھا']


## 6. Stemming

Chop affixes with rules to a crude root (not always a real word). Fast, good for rough grouping.
Urdu inflects for gender/number/tense, so it needs its own suffix rules.

In [8]:
if HAS_NLTK:
    from nltk.stem import PorterStemmer
    ps = PorterStemmer()
    print('porter:', [ps.stem(w) for w in ['running','studies','happiness','organization']])
else:
    def naive_stem(w):
        for suf in ('ing','ed','es','s'):
            if w.endswith(suf) and len(w) - len(suf) >= 3:
                return w[:-len(suf)]
        return w
    print('naive :', [naive_stem(w) for w in ['running','studies','plays','walked']])

porter: ['run', 'studi', 'happi', 'organ']


In [9]:
# A simple Urdu suffix stemmer (demo)
def urdu_stem(w):
    for suf in ('\u062a\u06cc','\u062a\u0627','\u0646\u0627','\u06cc\u0648\u06ba','\u06d2','\u06cc','\u0627'):
        if w.endswith(suf) and len(w) - len(suf) >= 2:
            return w[:-len(suf)]
    return w

forms = ['\u0644\u06a9\u06be\u062a\u0627','\u06a9\u06be\u06cc\u0644\u062a\u06cc','\u0644\u0691\u06a9\u06d2','\u067e\u0691\u06be\u062a\u0627']
print({w: urdu_stem(w) for w in forms})

{'لکھتا': 'لکھ', 'کھیلتی': 'کھیل', 'لڑکے': 'لڑک', 'پڑھتا': 'پڑھ'}


## 7. Lemmatization

Map a word to its dictionary form (a real word), using vocabulary and part of speech:
`studies → study`, `better → good`, `was → be`. More accurate than stemming, slower, and needs `nltk` + WordNet here.

In [10]:
if HAS_NLTK:
    try:
        from nltk.stem import WordNetLemmatizer
        wnl = WordNetLemmatizer()
        print(wnl.lemmatize('studies'), '|', wnl.lemmatize('better', pos='a'), '|', wnl.lemmatize('running', pos='v'))
    except Exception as e:
        print('WordNet data unavailable:', e)
else:
    print('Lemmatization needs nltk + wordnet. Install with:  pip install nltk  then nltk.download(\'wordnet\')')

study | good | run


## 8. Full pipeline, end to end

Now chain the steps into one function for each language and run them on a messy input.

In [11]:
def preprocess_en(text):
    text = clean(text).lower()
    toks = re.findall(r'[a-z]+', text)              # tokenize + drop punctuation/numbers
    toks = [t for t in toks if t not in EN_STOP]    # stop words
    if HAS_NLTK:
        try:
            from nltk.stem import WordNetLemmatizer
            wnl = WordNetLemmatizer()
            toks = [wnl.lemmatize(t, pos='v') for t in toks]
        except Exception:
            pass
    return toks

def preprocess_ur(text):
    toks = urdu_tokenize(text)                      # normalize + tokenize
    toks = [t for t in toks if t not in UR_STOP]    # stop words
    toks = [urdu_stem(t) for t in toks]             # stem
    return toks

print('EN:', preprocess_en('<p>The cats are RUNNING & playing!!! Visit http://x.com</p>'))
print('UR:', preprocess_ur('\u0644\u0691\u06a9\u06d2 \u0645\u06cc\u062f\u0627\u0646 \u0645\u06cc\u06ba \u06a9\u06be\u06cc\u0644 \u0631\u06c1\u06d2 \u062a\u06be\u06d2'))

EN: ['cat', 'run', 'play', 'visit']
UR: ['لڑک', 'میدان', 'کھیل']


## 9. From text to numbers: one-hot encoding

Build a vocabulary (each unique token gets an index), then represent each word as a vector that is all zeros
except a single `1` at its index. The math is identical for English and Urdu — once we have indices, script no longer matters.

In [12]:
vocab = ['cat', 'sat', 'mat', 'dog']
idx = {w: i for i, w in enumerate(vocab)}

def one_hot(word, vocab_index):
    v = [0] * len(vocab_index)
    v[vocab_index[word]] = 1
    return v

for w in vocab:
    print(f'{w:4s}', one_hot(w, idx))

cat  [1, 0, 0, 0]
sat  [0, 1, 0, 0]
mat  [0, 0, 1, 0]
dog  [0, 0, 0, 1]


In [13]:
# The same thing for Urdu tokens
uvocab = ['\u0644\u0691\u06a9', '\u0645\u06cc\u062f\u0627\u0646', '\u06a9\u06be\u06cc\u0644']  # larka, maidan, khel
uidx = {w: i for i, w in enumerate(uvocab)}
for w in uvocab:
    print(w, one_hot(w, uidx))

لڑک [1, 0, 0]
میدان [0, 1, 0]
کھیل [0, 0, 1]


In [14]:
# One-hot with scikit-learn
if HAS_SK and HAS_NUMPY:
    words = np.array(['cat', 'sat', 'mat', 'dog']).reshape(-1, 1)
    enc = OneHotEncoder(sparse_output=False, dtype=int)
    M = enc.fit_transform(words)
    print('categories:', list(enc.categories_[0]))
    print(M)
else:
    print('sklearn/numpy not available in this environment.')

categories: [np.str_('cat'), np.str_('dog'), np.str_('mat'), np.str_('sat')]
[[1 0 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [0 1 0 0]]


## 10. Bag of Words & TF-IDF

**Bag of Words** counts each vocabulary word per document (order ignored).
**TF-IDF** weights each word by how distinctive it is: frequent-everywhere words (stop words) get pushed toward zero,
rare-but-present words get boosted. TF-IDF is the strong classic baseline.

In [15]:
docs = ['cat sat cat', 'dog sat', 'cat ate']

if HAS_SK:
    cv = CountVectorizer()
    X = cv.fit_transform(docs)
    print('vocab :', list(cv.get_feature_names_out()))
    print('counts:')
    print(X.toarray())
    tf = TfidfVectorizer()
    T = tf.fit_transform(docs)
    print('tf-idf:')
    print(T.toarray().round(2))
else:
    vocab = sorted({w for d in docs for w in d.split()})
    print('vocab :', vocab)
    for d in docs:
        c = Counter(d.split())
        print(d, '->', [c[w] for w in vocab])

vocab : ['ate', 'cat', 'dog', 'sat']
counts:
[[0 2 0 1]
 [0 0 1 1]
 [1 1 0 0]]
tf-idf:
[[0.   0.89 0.   0.45]
 [0.   0.   0.8  0.61]
 [0.8  0.61 0.   0.  ]]


## 11. Where this goes next

One-hot / BoW / TF-IDF are **sparse** and carry no meaning — `cat` and `dog` are as far apart as `cat` and `mat`.
**Embeddings** (Word2Vec, GloVe, fastText, BERT) fix this with short dense vectors where similar words sit close together.
`fastText` and multilingual BERT work well for Urdu because they use subword information.

But every one of them still starts from the clean tokens produced above.

## 12. Exercises

1. Change `preprocess_en` to use stemming instead of lemmatization and compare the vocabularies.
2. Add three more Urdu stop words to `UR_STOP` and re-run Section 8.
3. Compute the one-hot (presence) vector of the whole sentence `'cat sat mat'` by summing word vectors.
4. For the three docs in Section 10, find which word has the highest TF-IDF in `'dog sat'` and explain why.
5. Take the Urdu sentence `مجھے کرکٹ کا کھیل پسند ہے` and run it through `preprocess_ur`, then one-hot encode the result.

---
# ✅ Solutions to Section 12 (Exercises)

Neeche har exercise ka solution hai. Har code cell ke upar comments mein likha hai ke kya ho raha hai.

## Exercise 1: Stemming vs Lemmatization (English)

`preprocess_en` mein lemmatizer use hua tha. Ab ek naya function `preprocess_en_stem` banate hain jo stemmer use karta hai, phir dono ki vocabulary compare karte hain.

In [16]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
ps = PorterStemmer()
wnl = WordNetLemmatizer()

def preprocess_en_stem(text):
    text = clean(text).lower()                     # clean + lowercase
    toks = re.findall(r'[a-z]+', text)             # tokenize
    toks = [t for t in toks if t not in EN_STOP]   # stop words hatao
    toks = [ps.stem(t) for t in toks]              # <-- STEMMING (rule se kaat do)
    return toks

def preprocess_en_lemma(text):
    text = clean(text).lower()
    toks = re.findall(r'[a-z]+', text)
    toks = [t for t in toks if t not in EN_STOP]
    toks = [wnl.lemmatize(t, pos='v') for t in toks]   # <-- LEMMATIZATION (dictionary form)
    return toks

sample = "The students are studying and the cats were running happily. Studies show better organization."
stem_out  = preprocess_en_stem(sample)
lemma_out = preprocess_en_lemma(sample)

print("Original :", sample)
print("Stemming     :", stem_out)
print("Lemmatization:", lemma_out)
print()
print("Stem vocab  :", sorted(set(stem_out)),  "-> size", len(set(stem_out)))
print("Lemma vocab :", sorted(set(lemma_out)), "-> size", len(set(lemma_out)))
print()
print("Sirf stemming mein aaye  :", sorted(set(stem_out) - set(lemma_out)))
print("Sirf lemmatization mein  :", sorted(set(lemma_out) - set(stem_out)))

Original : The students are studying and the cats were running happily. Studies show better organization.
Stemming     : ['student', 'studi', 'cat', 'were', 'run', 'happili', 'studi', 'show', 'better', 'organ']
Lemmatization: ['students', 'study', 'cat', 'be', 'run', 'happily', 'study', 'show', 'better', 'organization']

Stem vocab  : ['better', 'cat', 'happili', 'organ', 'run', 'show', 'student', 'studi', 'were'] -> size 9
Lemma vocab : ['be', 'better', 'cat', 'happily', 'organization', 'run', 'show', 'students', 'study'] -> size 9

Sirf stemming mein aaye  : ['happili', 'organ', 'student', 'studi', 'were']
Sirf lemmatization mein  : ['be', 'happily', 'organization', 'students', 'study']


**Nateeja:** Stemming kabhi kabhi aisay words banata hai jo asal mein words hi nahi hote (jaise `studi`, `happili`, `organ`). Lemmatization real dictionary word deta hai (jaise `study`, `happily`). Ek aur farq: `were` ko stemmer ne waisa hi chor diya, lekin lemmatizer ne usay `be` bana diya. Stemming tez hai, lemmatization zyada accurate.

## Exercise 2: Urdu mein 3 naye stop words add karna

Hum 3 common Urdu words add kar rahe hain: **بھی** (bhi), **کہ** (ke/that), **تو** (to). Phir Section 8 dobara run karte hain.

In [17]:
# Pehle ki list ka size
print("Pehle UR_STOP size:", len(UR_STOP))

# 3 naye stop words add karo
UR_STOP.update({'بھی', 'کہ', 'تو'})
print("Ab UR_STOP size   :", len(UR_STOP))
print()

# Section 8 wali original sentence (is mein naye words nahi the, isliye result same aayega)
orig = 'لڑکے میدان میں کھیل رہے تھے'
print("Original sentence :", preprocess_ur(orig))

# Nayi sentence jisme naye stop words hain: "woh bhi jaanta hai ke larke maidan mein khel rahe the to"
test = 'وہ بھی جانتا ہے کہ لڑکے میدان میں کھیل رہے تھے تو'
print("Naye words ke baghair (token list):", urdu_tokenize(test))
print("Naye stop words ke saath (result) :", preprocess_ur(test))

Pehle UR_STOP size: 17
Ab UR_STOP size   : 20

Original sentence : ['لڑک', 'میدان', 'کھیل']
Naye words ke baghair (token list): ['وہ', 'بھی', 'جانتا', 'ہے', 'کہ', 'لڑکے', 'میدان', 'میں', 'کھیل', 'رہے', 'تھے', 'تو']
Naye stop words ke saath (result) : ['جان', 'لڑک', 'میدان', 'کھیل']


**Nateeja:** `بھی`, `کہ` aur `تو` ab pipeline se nikal gaye. Section 8 ki original sentence mein ye words the hi nahi, isliye us ka output change nahi hua. Isi liye hum ne ek nayi sentence bhi test ki.

## Exercise 3: Poori sentence ka one-hot (presence) vector

Sentence: `'cat sat mat'`. Har word ka one-hot vector banao aur sab ko **jama (sum)** kar do.

In [18]:
import numpy as np

vocab = ['cat', 'sat', 'mat', 'dog']
idx = {w: i for i, w in enumerate(vocab)}

sentence = 'cat sat mat'
words = sentence.split()

vectors = [np.array(one_hot(w, idx)) for w in words]
for w, v in zip(words, vectors):
    print(f'{w:4s}', v)

total = np.sum(vectors, axis=0)
print()
print("Sum (poori sentence):", total)
print("Vocab order         :", vocab)

cat  [1 0 0 0]
sat  [0 1 0 0]
mat  [0 0 1 0]

Sum (poori sentence): [1 1 1 0]
Vocab order         : ['cat', 'sat', 'mat', 'dog']


**Nateeja:** `[1 1 1 0]`. Matlab `cat`, `sat`, `mat` sentence mein hain (1) aur `dog` nahi hai (0). Isay *presence vector* kehte hain.

## Exercise 4: `'dog sat'` mein sab se zyada TF-IDF kis word ka hai?

In [19]:
docs = ['cat sat cat', 'dog sat', 'cat ate']

tf = TfidfVectorizer()
T = tf.fit_transform(docs)
names = tf.get_feature_names_out()
idf = dict(zip(names, tf.idf_))

row = T[1].toarray()[0]          # doc number 1 = 'dog sat'
print("Words      :", list(names))
print("TF-IDF row :", row.round(3))
print()
for w, v in zip(names, row):
    if v > 0:
        print(f"{w}: tf-idf = {v:.3f}   (idf = {idf[w]:.3f})")

best = names[row.argmax()]
print("\nSab se zyada TF-IDF wala word:", best)

# Document frequency: kitne documents mein har word hai
for w in ['dog', 'sat']:
    df = sum(w in d.split() for d in docs)
    print(f"'{w}' {df} document(s) mein aaya (total {len(docs)})")

Words      : ['ate', 'cat', 'dog', 'sat']
TF-IDF row : [0.    0.    0.796 0.605]

dog: tf-idf = 0.796   (idf = 1.693)
sat: tf-idf = 0.605   (idf = 1.288)

Sab se zyada TF-IDF wala word: dog
'dog' 1 document(s) mein aaya (total 3)
'sat' 2 document(s) mein aaya (total 3)


**Jawab:** `dog` ka TF-IDF sab se zyada hai.

**Wajah:** Dono words (`dog`, `sat`) `'dog sat'` mein ek ek baar aaye, yaani TF barabar hai. Farq IDF ka hai. `sat` 3 mein se **2** documents mein hai (common hai), isliye uska weight kam hota hai. `dog` sirf **1** document mein hai (rare hai), isliye uska weight zyada hota hai. TF-IDF rare lekin important words ko upar utha deta hai.

## Exercise 5: Urdu sentence `مجھے کرکٹ کا کھیل پسند ہے` ka preprocess + one-hot

In [20]:
s = 'مجھے کرکٹ کا کھیل پسند ہے'

print("Step 1: normalize + tokenize :", urdu_tokenize(s))
tokens_no_stop = [t for t in urdu_tokenize(s) if t not in UR_STOP]
print("Step 2: stop words hatao     :", tokens_no_stop)
final = preprocess_ur(s)
print("Step 3: stemming (final)     :", final)

# One-hot encode final tokens
uvocab = sorted(set(final))
uidx = {w: i for i, w in enumerate(uvocab)}
print()
print("Vocabulary:", uvocab)
for w in final:
    print(w, one_hot(w, uidx))

Step 1: normalize + tokenize : ['مجھے', 'کرکٹ', 'کا', 'کھیل', 'پسند', 'ہے']
Step 2: stop words hatao     : ['مجھے', 'کرکٹ', 'کھیل', 'پسند']
Step 3: stemming (final)     : ['مجھ', 'کرکٹ', 'کھیل', 'پسند']

Vocabulary: ['مجھ', 'پسند', 'کرکٹ', 'کھیل']
مجھ [1, 0, 0, 0]
کرکٹ [0, 0, 1, 0]
کھیل [0, 0, 0, 1]
پسند [0, 1, 0, 0]


**Nateeja:** `کا` aur `ہے` stop words thay, wo hat gaye. Baqi 4 tokens bache aur har ek ko ek one-hot vector mila (4 length ka, jis mein sirf ek `1` hai).

**Dhyan dein:** `مجھے` (mujhe) ke aakhir se stemmer ne `ے` kaat diya aur woh `مجھ` ban gaya. Ye is simple demo stemmer ki kamzori hai. Asal mein `مجھے` khud ek stop word hai, isay stop-word list mein daalna behtar hota.